# Focus Coach — browser demo (Gradio)

Runs the Focus Coach pipeline (YOLO26n head + phone detection, rules, time smoothing, session report) in your browser, with no install.

**How to use:** Runtime → Change runtime type → **T4 GPU** (a CPU works but is slower) → edit only the config cell → **Run all** → click the public link printed at the end.
The link is temporary and stops working when this Colab session ends. If the live webcam feed is laggy, use the **Video** tab.

> **Note:** this notebook has not been run on a real Colab runtime yet. The demo code itself was tested locally; the steps that only exist in Colab (Drive, file upload, share link) were not.

**Config.** The only cell you may need to edit: your repo, and where the trained model is.

In [ ]:
REPO_URL = "https://github.com/majallen3690/focus-coach"  # EDIT: the GitHub repo that holds this project
MODEL_PATH = "/content/drive/MyDrive/focus-coach/runs/yolo26n_fpi/weights/best.pt"  # where Phase 2 saved the model on Drive
USE_DRIVE = True  # False: upload best.pt by hand in the model cell instead

**Runtime check.** Prints the GPU name. A CPU is fine for the Video and Image tabs, but the live tab will be slower.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. The demo works on a CPU but is slower. For a GPU: Runtime > Change runtime type > T4 GPU.")

**Get the code.** Clones your repo. If it is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left bar → Add new secret → turn on *Notebook access*). The token needs read access to the repo.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/focus-coach")
assert "YOUR-USERNAME" not in REPO_URL, "Edit REPO_URL in the config cell first."

if not REPO.exists():
    url = REPO_URL
    try:
        from google.colab import userdata

        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:  # no GITHUB_TOKEN secret: assume the repo is public
        pass
    done = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
    if done.returncode:  # print git's message without the token that is inside the URL
        raise RuntimeError(done.stderr.replace(url, REPO_URL))
os.chdir(REPO)
print("repo ready at", REPO)

**Get the model.** Copies `best.pt` from Drive, or shows an upload button when `USE_DRIVE = False`. It is saved as `models/best.pt`.

In [ ]:
import shutil

(REPO / "models").mkdir(exist_ok=True)
BEST = REPO / "models" / "best.pt"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    assert Path(MODEL_PATH).exists(), f"No model at {MODEL_PATH}. Fix MODEL_PATH in the config cell, or set USE_DRIVE = False."
    shutil.copy(MODEL_PATH, BEST)
else:
    from google.colab import files

    print("Choose your best.pt file:")
    uploaded = files.upload()
    assert uploaded, "No file was uploaded."
    shutil.move(next(iter(uploaded)), BEST)
print("model ready:", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")

**Install** the pinned requirements and Gradio. If Colab asks to restart the runtime after this cell, restart and Run all again; nothing is lost.

In [ ]:
!pip install -q -r requirements.txt -r requirements-demo.txt

**Smoke test.** Runs the model on one example photo and checks the answer. Stops here with a clear message if something is wrong, before the demo starts.

In [ ]:
import sys

sys.path.insert(0, str(REPO))
from app.pipeline import FramePipeline
from ml import rules
import cv2

cfg = rules.load_config()
pipeline = FramePipeline(cfg, weights=BEST)
example = REPO / "demo/examples/images/03_on_phone.jpg"
result = pipeline.process(cv2.imread(str(example)), "off")
for d in result.dets:
    print(f"  {d['cls']:5s} {d['conf']:.2f}")
assert result.state == rules.ON_PHONE, f"Smoke test failed: expected ON_PHONE for {example.name}, got {result.state}. Is best.pt the right model?"
print("Smoke test passed: the model sees the phone and the rules say", result.state)

**Launch.** Starts the demo and prints the public link. Open it, allow the camera on the Live tab, or use the Video / Image tabs.

In [ ]:
from demo.gradio_app import build_demo

demo = build_demo(model_path=BEST)
_, local_url, share_url = demo.queue().launch(share=True, prevent_thread_lock=True)
from IPython.display import HTML, display

display(HTML(f'<h2>Open the demo: <a href="{share_url}" target="_blank">{share_url}</a></h2>'
             "<p>This link is temporary and stops working when this Colab session ends. Frames are processed in this session and not stored.</p>"))
demo.block_thread()